# 02 — Graph Analysis
**Centrality + Community Detection**

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

plt.style.use('dark_background')
%matplotlib inline

## 1. Build Graph

In [ ]:
from src.graph_builder import load_processed, build_graph
from src.centrality import build_centrality_table, print_top_n

nodes, edges = load_processed()
G = build_graph(nodes, edges)
print(f'G: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')

## 2. Centrality Analysis

In [ ]:
cent_df = build_centrality_table(G)
print_top_n(cent_df, n=15)

In [ ]:
# Heatmap of top 20
metrics = ['degree_centrality','betweenness_centrality',
           'closeness_centrality','eigenvector_centrality','pagerank']
top20 = cent_df.reset_index(drop=True).head(20).set_index('name')[metrics]
top20_norm = (top20 - top20.min()) / (top20.max() - top20.min() + 1e-9)

fig, ax = plt.subplots(figsize=(10, 10))
sns.heatmap(top20_norm, annot=True, fmt='.2f', cmap='YlOrRd',
            linewidths=0.4, ax=ax)
ax.set_title('Centrality Heatmap — Top 20 Characters')
plt.tight_layout(); plt.show()

## 3. Community Detection — Louvain

In [ ]:
from src.communities import (
    detect_louvain, community_summary, compute_modularity
)

partition = detect_louvain(G)
comm_sum  = community_summary(G, partition)
mod       = compute_modularity(G, partition)

print(f'Modularity: {mod:.4f}')
print(comm_sum.to_string(index=False))

## 4. Community Visualization

In [ ]:
from src.visualization import plot_community_network
plot_community_network(G, partition)
from IPython.display import Image
Image('outputs/figures/network_communities.png')

## 5. Interactive Network

In [ ]:
from src.visualization import plot_pyvis_network
plot_pyvis_network(G, cent_df, partition)

from IPython.display import IFrame
IFrame('outputs/figures/network_interactive.html', width=900, height=600)

## 6. PageRank vs Degree Scatter

In [ ]:
import plotly.express as px
df = cent_df.reset_index(drop=True).head(30)
fig = px.scatter(
    df, x='degree_centrality', y='pagerank',
    text='name', size='centrality_score',
    color='betweenness_centrality',
    color_continuous_scale='Plasma',
    title='PageRank vs Degree Centrality',
    template='plotly_dark'
)
fig.update_traces(textposition='top center')
fig.show()